# 🛡️ Validación robusta de entrada
*(versión explicada paso a paso)*

---

## El problema

En todos los ejemplos del Tema 3 y los primeros del Tema 4, **asumimos que el usuario teclea bien**. Pero, ¿qué pasa si el cálculo de la nota recibe `'cinco'` en lugar de `5`? El programa **se rompe**.

Un programa profesional **nunca confía** en que la entrada sea correcta: la **valida** y, si no lo es, vuelve a pedir el dato. Y para hacer eso necesitamos las dos construcciones estrella del Tema 4: el **bucle** (`while`) y el **manejo de excepciones** (`try-except`).

## Dos cosas pueden ir mal

Cuando le pides al usuario un número entero entre 1 y 100, puede equivocarse de **dos maneras distintas**:

1. **Error de tipo**: teclea algo que no es un número (`'hola'`, `'3.14'`, una línea en blanco…). Esto provoca un `ValueError` cuando intentamos `int(...)`.
2. **Error de rango**: teclea un número entero correcto, pero fuera del rango permitido (por ejemplo `-5` o `200`). Esto **no es una excepción de Python**: somos nosotros los que decidimos que no nos vale.

Cada uno requiere una técnica distinta:

| Tipo de error | Detección |
| :--- | :--- |
| Error de tipo | `try-except ValueError` |
| Error de rango | `if` |


## Paso 1: Estructura básica con `while True`

Como no sabemos cuántas veces se va a equivocar el usuario, usamos un bucle infinito y salimos con `break` cuando todo esté bien.

In [ ]:
# (no ejecutar esta celda, es solo el esqueleto)
# while True:
#     entrada = input('Introduce un número: ')
#     # ... validar ...
#     # si es válido → break

> 💡 En el [Tema 4 (teoría)](../teoria/T4_ICC.md) se desaconseja `while True` en general, pero **este es uno de los casos legítimos** donde resulta natural: "hazlo hasta que algo bueno ocurra". Es justamente el patrón que en otros lenguajes se hace con `do-while`. No obstante, es evitable completamente, como veremos en el Ejemplo 4. 

## Paso 2: Capturar el error de tipo

Envolvemos la conversión `int(...)` con `try-except`. Si falla, mostramos un mensaje y volvemos a empezar con `continue`:

In [ ]:
while True:
    entrada = input('Introduce un número entero entre 1 y 100: ')
    try:
        n = int(entrada)
    except ValueError:
        print(f'❌ "{entrada}" no es un entero válido.')
        continue
    if 1 <= n <= 100:
        break
    else:
        print(f'❌ {n} está fuera del rango [1, 100].')

print(f'✅ Has introducido el número {n}. ¡Gracias!')

> 🎯 **Prueba a teclear**: `42`, `hola`, `3.14`, `-5`, `200`, una línea en blanco. Observa cómo cada caso se gestiona limpiamente, sin romperse.

Vamos a ver otra alternativa, que evita el `while True`:

In [ ]:
entrada_valida = False
while not entrada_valida:
    entrada = input('Introduce un número entero entre 1 y 100: ')
    try:
        n = int(entrada)
    except ValueError:
        print(f'❌ "{entrada}" no es un entero válido.')
    else:
        if 1 <= n <= 100:
            entrada_valida = True
        else:
            print(f'❌ {n} está fuera del rango [1, 100].')
print(f'✅ Has introducido el número {n}. ¡Gracias!')

🧐 **¿Cuál te parece más interpretable?¿Cuál crees que es más eficiente?**

La interpretablidad es cuestionable, el primer bloque se alinea más con la filosofía del lenguaje Python (es lo que se suele llamar código pythonico). No obstante, los puristas y rigurosos de la **programación estructurada**, prefieren la segunda, por ser **formalmente más correcto**, y verán el primer código, en el mejor de los casos, con recelo, y en el peor, como una **mala práctica** o **pecado menor**. 

En cuanto a la eficiencia, la diferencia es mínima, quizás es mejor la primera opción porque evita el uso de una variable `entrada_valida` sobre la que hay que hacer modificaciones y comprobaciones.

## Paso 3: Lo que NO debes hacer

Un error muy común al empezar es capturar **cualquier** excepción con un `except` sin tipo:

```python
try:
    n = int(entrada)
except:           # ❌ captura TODO
    pass
```

Esto es como **esconder la basura debajo de la alfombra**: silencias incluso los errores que no esperabas, y nunca te enteras de que algo va mal en otra parte del programa.

**Siempre** especifica el tipo concreto:

```python
except ValueError:   # ✅ solo capturamos esto, lo demás se propaga
```

Otro error común es eliminar el bloque `else` incluyendo dentro del bloque `try` todo el código que se debe ejecutar. Existe un principio en programación **"Catch as little as possible" (Atrapa lo menos posible)**, que establece que el bloque `try` debe contener **únicamente** la línea de código que temes que falle.

‼️ Ejemplo de **Mal hábito**:

```python
entrada = input("¿En qué año naciste?: ")

try:
    # 1. ESTO ES LO ÚNICO PELIGROSO: Puede fallar si escriben "hola"
    ano = int(entrada)
    # 2. OPERACIONES SEGURAS: Solo se ejecutan si el 'int' funcionó
    edad = 2026 - ano
    print(f"Tienes {edad} años.")
    print("Eres mayor de edad." if edad >= 18 else "Eres menor de edad.")
except ValueError:
    print("❌ Eso no es un año válido.")
```

✅ Ejemplo de **Buen hábito**:

```python
entrada = input("¿En qué año naciste?: ")

try:
    # 1. LO ÚNICO PELIGROSO, lo que puede fallar se mete en el bloque try
    ano = int(entrada)
except ValueError:
    print("❌ Eso no es un año válido.")
else:
    # 2. Las operaciones que solo se ejecutan si lo que está en el bloque try no dio problemas
    edad = 2026 - ano
    print(f"Tienes {edad} años.")
    print("Eres mayor de edad." if edad >= 18 else "Eres menor de edad.")
```

## 🎯 Conceptos del Tema 4 que has practicado

* ✅ **Manejo de excepciones** con `try-except`.
* ✅ Especificar siempre el **tipo concreto** de excepción.
* ✅ Distinguir entre errores de **tipo** (excepción) y errores de **rango** (validación con `if`).
* ✅ Patrón `while True` con `break` como salida controlada.
* ✅ Uso de `continue` para reiniciar el ciclo.

## 🚀 Para reflexionar

* Aplica este patrón a los ejemplos anteriores (farmacología y finanzas). Verás cómo los programas se vuelven mucho más robustos.
* ¿Cómo lo cambiarías para que el usuario pueda **rendirse** tecleando `'salir'`?
* ¿Cómo limitarías el número de intentos a 3?